# Turbopuffer RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job using Turbopuffer as the corpus backend:

1. **Point to dataset** - Chunk and upload documents to a Turbopuffer namespace
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Configure the Turbopuffer search environment
4. **Run training job** - Launch the training

## Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev] turbopuffer

In [ ]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
repo_root = next((p for p in candidate_roots if (p / "src" / "cgft").exists()), cwd)
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
import cgft
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
# Configuration

# Turbopuffer credentials — get your API key at https://turbopuffer.com/
TURBOPUFFER_API_KEY = ""
TURBOPUFFER_REGION = ""
NAMESPACE = ""

# CGFT API key — create one at https://app.cgft.io/account/api-keys
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# LLM API credentials — used by CgftPipeline for QA generation, filtering, and refinement
LLM_API_KEY = ""
LLM_BASE_URL = ""

# Judge LLM credentials — used at reward time to evaluate correctness + citation
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-non-reasoning"

# Dataset configuration
# This should point to a local directory containing the documents you want to use.
# Go to docs.cgft.io/rag/synthetic_datagen for sample documents you can use.
DOCS_PATH = "./samples/posthog/docs/"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 30
OUTPUT_DIR = "outputs/tpuf_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = None  # e.g. "posthog-tpuf-v1"
EXPERIMENT_PREFIX = None  # e.g. "posthog-tpuf"

# Optional: provide Azure OpenAI credentials to store vector embeddings alongside BM25.
# Set EMBED_API_KEY to None to use BM25 full-text search only (no vectors stored).
EMBED_API_KEY = ""      # e.g. "4T3rpNzE..."
EMBED_ENDPOINT = ""     # e.g. "https://my-resource.cognitiveservices.azure.com/"
EMBED_DEPLOYMENT = ""   # e.g. "text-embedding-3-small"
EMBED_API_VERSION = ""

## Step 1: Point to Dataset

Chunk your documents and upload to a Turbopuffer namespace in one line.

> If you already have a populated namespace you want to use, skip this step.

In [ ]:
from cgft.corpus.turbopuffer.source import TpufChunkSource

source = TpufChunkSource(api_key=TURBOPUFFER_API_KEY, namespace=NAMESPACE, region=TURBOPUFFER_REGION)

# Optional: build an embedding function to store vectors alongside BM25.
# Remove this block (or leave EMBED_API_KEY=None above) for BM25-only.
embed_fn = None
if EMBED_API_KEY:
    from openai import AzureOpenAI
    _openai = AzureOpenAI(api_key=EMBED_API_KEY, api_version=EMBED_API_VERSION, azure_endpoint=EMBED_ENDPOINT)
    def embed_fn(texts):
        response = _openai.embeddings.create(model=EMBED_DEPLOYMENT, input=texts)
        return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]

source.populate_from_folder(DOCS_PATH, embed_fn=embed_fn)

## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the Turbopuffer corpus from Step 1.

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    EntityExtractionLLMConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LinkerConfig,
    LLMDirectGenerationConfig,
    LLMEnvGenerationConfig,
    LLMGuidedLinkerConfig,
    OutputConfig,
    PlatformConfig,
    RefinementConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
    TransformationConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name="test", min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
    ),
    targets=TargetsConfig(
            total_samples=TOTAL_SAMPLES,
            qa_type_distribution ={
                "lookup": 1,
                "co_located_multi_hop": 0,
                "cross_document_multi_hop": 0,
                "sequential_reasoning": 0,
                "synthesis": 0.0
            }),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)

In [ ]:
# Reuse the already-loaded corpus source from Step 1.
import importlib
from pprint import pprint

import cgft.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import cgft.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft_pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft_pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]


## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import cgft
from cgft.envs.tpuf_search_env import TpufSearchEnv
from cgft.trainer.pipeline import train

experiment_id = train(
    env_class=TpufSearchEnv,
    env_args={
        "turbopuffer_api_key": TURBOPUFFER_API_KEY,
        "namespace": NAMESPACE,
        "region": TURBOPUFFER_REGION,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    pip_dependencies=["turbopuffer", "openai"],
    experiment_name=EXPERIMENT_NAME,
)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse